In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/seyhaofficial/voxcpm2/config.json
/kaggle/input/datasets/seyhaofficial/voxcpm2/audiovae.pth
/kaggle/input/datasets/seyhaofficial/voxcpm2/README.md
/kaggle/input/datasets/seyhaofficial/voxcpm2/tokenizer.json
/kaggle/input/datasets/seyhaofficial/voxcpm2/tokenizer_config.json
/kaggle/input/datasets/seyhaofficial/voxcpm2/gitattributes
/kaggle/input/datasets/seyhaofficial/voxcpm2/model.safetensors
/kaggle/input/datasets/seyhaofficial/voxcpm2/special_tokens_map.json
/kaggle/input/datasets/seyhaofficial/voxcpm2/tokenization_voxcpm2.py


In [2]:
!pip install voxcpm --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 1.3 MB/s eta 0:00:00


In [3]:
#@title ✅ Verify GPU & VRAM
import torch
assert torch.cuda.is_available(), "❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")

✅ GPU: Tesla T4  |  VRAM: 15.6 GB


In [4]:
# @title 🚀 ២. បើកប្រើប្រាស់ម៉ូដែលពី Kaggle Dataset
import os
import torch
import torch._dynamo

# បិទកូដដែលទើសទែង
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True
os.environ["USE_MODELSCOPE"] = "False"

# ហៅបណ្ណាល័យដែលទើបដំឡើងមុននេះមកប្រើ
from voxcpm import VoxCPM

print("កំពុងបើកម៉ូដែលពី Dataset VOXCPM2 ក្នុង Kaggle...")

# ហៅចេញពីផ្លូវថតឯកសារ Dataset នៅក្នុង Kaggle
model = VoxCPM.from_pretrained(
    "/kaggle/input/datasets/seyhaofficial/voxcpm2",  # ← ដូរទីតាំងមក Kaggle Dataset វិញ
    load_denoiser=False,
    optimize=False
)

SAMPLE_RATE = model.tts_model.sample_rate
print("✅ ម៉ូដែលបានបើកដំណើរការជោគជ័យ!")

កំពុងបើកម៉ូដែលពី Dataset VOXCPM2 ក្នុង Kaggle...


voxcpm_model_path: /kaggle/input/datasets/seyhaofficial/voxcpm2, zipenhancer_model_path: None, enable_denoiser: False
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /kaggle/input/datasets/seyhaofficial/voxcpm2/audiovae.pth
Running on device: cuda, dtype: bfloat16
Loading model from safetensors: /kaggle/input/datasets/seyhaofficial/voxcpm2/model.safetensors


✅ ម៉ូដែលបានបើកដំណើរការជោគជ័យ!


Loaded VoxCPM2Model


In [5]:
!pip install fastapi uvicorn python-multipart nest-asyncio
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠴

In [6]:
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
import uvicorn
import nest_asyncio
import soundfile as sf
import tempfile
import threading
import os

app = FastAPI(title="VoxCPM2 Voice Cloning API")

@app.post("/clone-voice/")
async def clone_voice_api(
    text: str = Form(..., description="អត្ថបទដែលចង់ឱ្យអាន"), 
    reference_audio: UploadFile = File(..., description="File សំឡេងដើម (WAV/MP3)")
):
    print(f"ទទួលបានសំណើ: {text}")
    
    # 1. Save file សំឡេងដែលទទួលបានពី App ជាបណ្ដោះអាសន្ន
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as temp_ref:
        temp_ref.write(await reference_audio.read())
        temp_ref_path = temp_ref.name

    # 2. ដំណើរការ Clone សំឡេង
    wav = model.generate(
        text=text,
        reference_wav_path=temp_ref_path,
        cfg_value=2.0,
        inference_timesteps=10,
    )

    # 3. Save លទ្ធផល និងលុប file បណ្ដោះអាសន្នចោល
    output_path = "cloned_result_api.wav"
    sf.write(output_path, wav, SAMPLE_RATE)
    os.remove(temp_ref_path)

    # 4. បញ្ជូន File សំឡេងត្រលប់ទៅកាន់ App វិញ
    return FileResponse(output_path, media_type="audio/wav", filename="cloned_voice.wav")

# អនុញ្ញាតឱ្យ FastAPI ដើរក្នុង Background របស់ Colab
nest_asyncio.apply()
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_api, daemon=True).start()
print("✅ API កំពុងដំណើរការនៅលើ Port 8000...")

✅ API កំពុងដំណើរការនៅលើ Port 8000...


In [7]:
!pip install pyngrok

INFO:     Started server process [23]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [8]:
from pyngrok import ngrok

# យកសញ្ញា # ออกពីមុខកូដនេះ ដើម្បីឱ្យវាទាញយក Token របស់អ្នកទៅ Register
ngrok.set_auth_token("3Iate69C7gzHcCE12bLtyh2An16_3HFmhJpB6QJiDBBLnq4Vx")

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://pod-moisten-sauciness.ngrok-free.dev" -> "http://localhost:8000"
